In [ ]:
# Dataset
# The manual names RealEstate_Prices.csv but does not specify a download source. You can use an open-source housing dataset as an alternative:
# - Kaggle – House Prices Dataset: https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data?utm_source=chatgpt.com
# - California Housing Dataset – Scikit-learn: https://scikit-learn.org/stable/datasets/real_world.html?utm_source=chatgpt.com#california-housing-dataset




# ============================================================
# EXPERIMENT NO. 4
# DATA WRANGLING ON REAL ESTATE MARKET
# Dataset: RealEstate_Prices.csv
# ============================================================

# STEP 1: Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder

from google.colab import files

# STEP 2: Upload the real estate dataset

print("Upload RealEstate_Prices.csv")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

# STEP 3: Load the dataset

df = pd.read_csv(file_name)

print("\n========== ORIGINAL DATASET ==========")
print(df.head())

# STEP 4: Explore the dataset

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Information:")
df.info()

print("\nStatistical Description:")
print(df.describe(include="all"))

# STEP 5: Clean column names
# Convert names to lowercase and replace spaces/special characters.

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("\nCleaned Column Names:")
print(df.columns.tolist())

# STEP 6: Remove duplicate records

print("\nDuplicate Records:", df.duplicated().sum())

df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

# STEP 7: Handle missing values

print("\n========== MISSING VALUES ==========")
print(df.isnull().sum())

# Identify numerical and categorical columns.

numeric_columns = df.select_dtypes(
    include=np.number
).columns

categorical_columns = df.select_dtypes(
    include=["object", "string"]
).columns

# Fill missing numerical values using median.

for column in numeric_columns:
    df[column] = df[column].fillna(
        df[column].median()
    )

# Fill missing categorical values using mode.

for column in categorical_columns:

    mode_value = df[column].mode()

    if not mode_value.empty:
        df[column] = df[column].fillna(mode_value[0])
    else:
        df[column] = df[column].fillna("Unknown")

# STEP 8: Standardize categorical values

for column in categorical_columns:

    df[column] = (
        df[column]
        .astype(str)
        .str.strip()
        .str.title()
    )

print("\nMissing Values After Cleaning:")
print(df.isnull().sum())

# STEP 9: Identify important real estate columns

# The code checks for common names used in real estate datasets.

def find_column(possible_names):

    for name in possible_names:
        if name in df.columns:
            return name

    return None


price_col = find_column([
    "price", "sale_price", "selling_price",
    "house_price", "property_price"
])

area_col = find_column([
    "area", "square_feet", "sqft",
    "size", "living_area"
])

location_col = find_column([
    "location", "neighborhood", "city",
    "locality", "region"
])

property_col = find_column([
    "property_type", "type", "house_type"
])

date_col = find_column([
    "date", "sale_date", "date_sold"
])

print("\n========== DETECTED COLUMNS ==========")
print("Price Column:", price_col)
print("Area Column:", area_col)
print("Location Column:", location_col)
print("Property Type Column:", property_col)
print("Date Column:", date_col)

# STEP 10: Convert price and area columns to numeric

for column in [price_col, area_col]:

    if column is not None:

        df[column] = pd.to_numeric(
            df[column].astype(str).str.replace(
                r"[$,]", "", regex=True
            ),
            errors="coerce"
        )

        df[column] = df[column].fillna(
            df[column].median()
        )

# Convert date column into datetime format.

if date_col is not None:

    df[date_col] = pd.to_datetime(
        df[date_col],
        errors="coerce"
    )

# STEP 11: Filter and subset data

filtered_df = df.copy()

# Example: Remove records with non-positive property prices.

if price_col is not None:

    filtered_df = filtered_df[
        filtered_df[price_col] > 0
    ]

print("\n========== FILTERED DATA ==========")
print(filtered_df.head())

# STEP 12: Perform feature engineering

# Calculate price per square foot if price and area exist.

if price_col is not None and area_col is not None:

    filtered_df["price_per_sqft"] = (
        filtered_df[price_col] /
        filtered_df[area_col].replace(0, np.nan)
    )

    print("\nPrice per Square Foot:")
    print(filtered_df["price_per_sqft"].head())

# STEP 13: Identify and handle outliers using IQR

numeric_columns = filtered_df.select_dtypes(
    include=np.number
).columns

print("\n========== OUTLIER DETECTION ==========")

for column in numeric_columns:

    Q1 = filtered_df[column].quantile(0.25)
    Q3 = filtered_df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = filtered_df[
        (filtered_df[column] < lower_limit) |
        (filtered_df[column] > upper_limit)
    ]

    print(column, ":", len(outliers), "outliers")

    # Cap extreme values instead of deleting records.

    filtered_df[column] = filtered_df[column].clip(
        lower=lower_limit,
        upper=upper_limit
    )

# STEP 14: Encode categorical variables

encoded_df = filtered_df.copy()

categorical_columns = encoded_df.select_dtypes(
    include=["object", "string", "category"]
).columns

for column in categorical_columns:

    encoder = LabelEncoder()

    encoded_df[column] = encoder.fit_transform(
        encoded_df[column].astype(str)
    )

print("\n========== ENCODED DATASET ==========")
print(encoded_df.head())

# STEP 15: Calculate average price by location

if price_col is not None and location_col is not None:

    avg_price_location = (
        filtered_df.groupby(location_col)[price_col]
        .mean()
        .sort_values(ascending=False)
    )

    print("\n========== AVERAGE PRICE BY LOCATION ==========")
    print(avg_price_location)

    # Bar chart for average property price by location.

    plt.figure(figsize=(10, 5))

    avg_price_location.plot(kind="bar")

    plt.title("Average Property Price by Location")
    plt.xlabel("Location")
    plt.ylabel("Average Price")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# STEP 16: Calculate average price by property type

if price_col is not None and property_col is not None:

    avg_price_type = (
        filtered_df.groupby(property_col)[price_col]
        .mean()
    )

    print("\n========== AVERAGE PRICE BY PROPERTY TYPE ==========")
    print(avg_price_type)

    plt.figure(figsize=(8, 5))

    avg_price_type.plot(kind="bar")

    plt.title("Average Price by Property Type")
    plt.xlabel("Property Type")
    plt.ylabel("Average Price")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# STEP 17: Visualize price distribution

if price_col is not None:

    plt.figure(figsize=(8, 5))

    sns.histplot(
        filtered_df[price_col],
        kde=True
    )

    plt.title("Distribution of Property Prices")
    plt.xlabel("Property Price")
    plt.ylabel("Frequency")
    plt.show()

# STEP 18: Visualize relationship between area and price

if price_col is not None and area_col is not None:

    plt.figure(figsize=(8, 5))

    sns.scatterplot(
        data=filtered_df,
        x=area_col,
        y=price_col
    )

    plt.title("Property Area vs Property Price")
    plt.xlabel("Area")
    plt.ylabel("Price")
    plt.show()

# STEP 19: Create correlation heatmap

numeric_columns = filtered_df.select_dtypes(
    include=np.number
).columns

if len(numeric_columns) >= 2:

    plt.figure(figsize=(10, 6))

    sns.heatmap(
        filtered_df[numeric_columns].corr(),
        annot=True,
        cmap="coolwarm",
        fmt=".2f"
    )

    plt.title("Real Estate Data Correlation Heatmap")
    plt.tight_layout()
    plt.show()

# STEP 20: Save the cleaned and transformed datasets

filtered_df.to_csv(
    "cleaned_real_estate_data.csv",
    index=False
)

encoded_df.to_csv(
    "encoded_real_estate_data.csv",
    index=False
)

print("\n========== EXPERIMENT COMPLETED ==========")
print("Cleaned dataset saved successfully.")
print("Encoded dataset saved successfully.")

# Download the cleaned dataset.

files.download("cleaned_real_estate_data.csv")
